# Emberline clean data rebuild — Stage 1: workspace and source inventory

Run this notebook in Google Colab with a standard CPU runtime. It creates a new versioned workspace in your Google Drive. Stage 1 performs source discovery only. Downloading, pixel analysis, calibration and frontend export are later stages with explicit checks.

Keep the existing frontend and old data snapshots intact. This notebook does not import old fire totals, models or validation scores. A clean workspace improves traceability; scientific accuracy must still be established by validation.

Pilot: disputed January 2001 and December 2023; controls January 2002 and December 2022. A small pilot catches archive, temporal-overlap and quality issues before a full historical download.

**Run cells in order.** If a cell fails, stop and share the error without passwords or tokens.


## 1. Install the notebook tools

No GPU is required. Record the installed versions below; the complete environment lock is saved to this run. If Colab asks for a runtime restart, restart and rerun from this cell.


In [3]:
%pip -q install earthaccess==0.19.0 requests pandas numpy pyarrow duckdb shapely h3 pyhdf pyproj scipy


## 2. Connect Google Drive

Approve the Google Drive connection when Colab asks. Earthdata authentication is a separate step.


In [4]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Create a fresh, versioned workspace

Each execution creates a separate run. To resume a run after a disconnect, set `RESUME_RUN` to the previously printed run ID. Raw files will remain separate from normalized tables, scientific results and web exports. Download working files to Colab's local disk first; copy verified files and checkpoints to Drive.


In [5]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, uuid, subprocess, sys
import importlib.metadata as metadata

RESUME_RUN = None  # Example: '20261114T120000Z_a1b2c3d4'; use only an existing run you created.
BASE = Path('/content/drive/MyDrive/emberline_clean_rebuild')
RUN_ID = RESUME_RUN or (datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8])
assert '/' not in RUN_ID and '..' not in RUN_ID
RUN = BASE / 'runs' / RUN_ID
if RESUME_RUN:
    assert RUN.is_dir(), 'Requested resume folder does not exist.'
else:
    RUN.mkdir(parents=True, exist_ok=False)
for folder in ['inventory', 'raw', 'normalized', 'coverage', 'validation', 'exports_staging', 'logs']:
    (RUN / folder).mkdir(exist_ok=True)
LOCAL = Path('/content/emberline_work') / RUN_ID
LOCAL.mkdir(parents=True, exist_ok=True)
packages = ['earthaccess','requests','pandas','numpy','pyarrow','duckdb','shapely','h3','pyhdf','pyproj','scipy']
versions = {name: metadata.version(name) for name in packages}
environment_path = RUN/'logs/environment.json'
if environment_path.exists():
    assert json.loads(environment_path.read_text())['packages'] == versions, 'Package versions changed: restore this run lock or create a new run.'
else:
    environment_path.write_text(json.dumps({'python':sys.version,'packages':versions}, indent=2))
if not (RUN/'logs/requirements.lock.txt').exists():
    (RUN/'logs/requirements.lock.txt').write_bytes(subprocess.check_output([sys.executable,'-m','pip','freeze']))
print('RUN_ID:', RUN_ID)
print('Drive workspace:', RUN)
print('Installed versions:', versions)


RUN_ID: 20261114T163518Z_5b1b1c54
Drive workspace: /content/drive/MyDrive/emberline_clean_rebuild/runs/20261114T163518Z_5b1b1c54
Installed versions: {'earthaccess': '0.19.0', 'requests': '2.32.4', 'pandas': '2.2.3', 'numpy': '2.1.3', 'pyarrow': '23.0.1', 'duckdb': '1.3.2', 'shapely': '2.2.0', 'h3': '4.5.0', 'pyhdf': '0.11.7', 'pyproj': '3.8.0', 'scipy': '1.16.3'}


## 4. Freeze definitions for the pilot

- Unit: fire-active H3 resolution-6 cell-days after sensor-specific daily deduplication. Raw detections remain available separately.
- Sensor identity, product version, acquisition time, native confidence and processing lineage remain attached to every source record.
- The bbox discovers candidate files; the approved regional polygon will determine spatial support in the processing stage.
- Archive completeness and usable observing conditions are separate quantities. Clouds do not establish absence of fire.
- All four categories × three display-confidence combinations must be emitted for every month. Native confidence is retained; display classes are not equivalent probabilities across sensors.
- Calibrated history, gap filling and local allocation require held-out validation before release.


In [6]:
CONFIG = {
    'region': 'lk',
    'bbox': [79.4, 5.8, 82.0, 10.0],
    'h3_resolution': 6,
    'unit': 'fire-active cell-days',
    'categories': ['veg','volcano','static','offshore'],
    'display_confidence': [1,2,3],
    'pilot_months': {
        '2001-01':'disputed', '2002-01':'control',
        '2022-12':'control', '2023-12':'disputed'
    },
    'production_export_enabled': False,
    'noaa20_fill_enabled': False,
}
config_path = RUN/'config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == CONFIG, 'Config changed: create a new run.'
else:
    config_path.write_text(json.dumps(CONFIG, indent=2))
print(json.dumps(CONFIG, indent=2))


{
  "region": "lk",
  "bbox": [
    79.4,
    5.8,
    82.0,
    10.0
  ],
  "h3_resolution": 6,
  "unit": "fire-active cell-days",
  "categories": [
    "veg",
    "volcano",
    "static",
    "offshore"
  ],
  "display_confidence": [
    1,
    2,
    3
  ],
  "pilot_months": {
    "2001-01": "disputed",
    "2002-01": "control",
    "2022-12": "control",
    "2023-12": "disputed"
  },
  "production_export_enabled": false,
  "noaa20_fill_enabled": false
}


## 5. Sign in to Earthdata for this Colab session

Enter credentials only in the interactive prompts. `persist=False` avoids writing a password file. Colab has its own session even if you have already signed in through a browser. Authentication is needed for later science downloads; public catalogue searches work without it.

Reference: https://earthaccess.readthedocs.io/en/stable/user/howto/authenticate/


In [7]:
import earthaccess
auth = earthaccess.login(strategy='interactive', persist=False)
assert auth.authenticated, 'Earthdata login did not succeed.'
print('Earthdata authentication succeeded. Credentials are not included in run metadata.')


Earthdata authentication succeeded. Credentials are not included in run metadata.


## 6. Discover the current standard-product collections

Daily L3 fire masks (`MOD14A1`, `MYD14A1`) support initial screening. Swath fire masks (`MOD14`, `MYD14`) and geolocation (`MOD03`, `MYD03`) are needed for acquisition-level reconciliation. Preserve the appropriate Collection 6.1 versions; do not silently substitute Collection 7 geolocation or NRT data.

This cell queries public NASA CMR metadata and saves the exact responses and hashes. Collection identifiers are discovered rather than guessed.


In [8]:
import requests, time
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()  # Catalogue-only session; no authentication headers added.
retry = Retry(total=4, backoff_factor=1, status_forcelist=[429,500,502,503,504])
session.mount('https://', HTTPAdapter(max_retries=retry))
CMR = 'https://cmr.earthdata.nasa.gov/search/'
query_log_path = RUN/'inventory/query_log.json'
query_log = json.loads(query_log_path.read_text()) if query_log_path.exists() else []

def cmr_page(endpoint, params, filename):
    response = session.get(CMR+endpoint, params=params, timeout=90)
    response.raise_for_status()
    raw = response.content
    digest = hashlib.sha256(raw).hexdigest()
    snapshot = RUN/'inventory'/(Path(filename).stem + '_' + digest + '.json')
    if snapshot.exists():
        assert snapshot.read_bytes() == raw
    else:
        snapshot.write_bytes(raw)
    query_log.append({'url':response.url, 'snapshot':snapshot.name, 'sha256':digest,
                      'retrieved_at_utc':datetime.now(timezone.utc).isoformat()})
    (RUN/'inventory/query_log.json').write_text(json.dumps(query_log, indent=2))
    return response.json(), response.headers

PRODUCTS = {
    'MOD14A1':('061','LPCLOUD'), 'MYD14A1':('061','LPCLOUD'),
    'MOD14':('061','LPCLOUD'), 'MYD14':('061','LPCLOUD'),
    'MOD03':('6.1','LAADS'), 'MYD03':('6.1','LAADS'),
}
COLLECTIONS = {}
for product,(version,provider) in PRODUCTS.items():
    payload,_ = cmr_page('collections.json',
        {'short_name':product,'version':version,'page_size':100}, f'{product}_collection.json')
    matches=[e for e in payload['feed']['entry'] if e.get('data_center')==provider]
    assert len(matches)==1, f'{product}: expected one standard collection; review metadata before proceeding.'
    COLLECTIONS[product]=matches[0]['id']
collection_path = RUN/'inventory/collections.json'
if collection_path.exists():
    assert json.loads(collection_path.read_text()) == COLLECTIONS, 'Collections changed: create a new run.'
else:
    collection_path.write_text(json.dumps(COLLECTIONS, indent=2))
print(COLLECTIONS)


{'MOD14A1': 'C2565791013-LPCLOUD', 'MYD14A1': 'C2565794059-LPCLOUD', 'MOD14': 'C2271754179-LPCLOUD', 'MYD14': 'C2278858993-LPCLOUD', 'MOD03': 'C1379767668-LAADS', 'MYD03': 'C1379841358-LAADS'}


## 7. Inventory the four pilot months

This fetches metadata, not science files. It paginates all results and checks CMR's hit count when supplied. Bbox results are candidates; clip footprints to the approved boundary later. File counts are not counts of usable observations or proof of complete acquisition coverage.

Aqua has no records in the two January pilot months because they precede its observing record. That absence must not be classified as a sensor outage.


In [9]:
import pandas as pd
all_entries = {}
summary = []
for month,role in CONFIG['pilot_months'].items():
    start = pd.Timestamp(month+'-01')
    end = start + pd.offsets.MonthBegin(1) - pd.Timedelta(seconds=1)
    temporal = f'{start:%Y-%m-%dT00:00:00Z},{end:%Y-%m-%dT%H:%M:%SZ}'
    for product,collection in COLLECTIONS.items():
        entries=[]; page=1; expected=None
        while True:
            payload,headers=cmr_page('granules.json',
                {'collection_concept_id':collection,'bounding_box':','.join(map(str,CONFIG['bbox'])),
                 'temporal':temporal,'page_size':200,'page_num':page},
                f'{product}_{month}_page{page}.json')
            if page==1 and headers.get('CMR-Hits') is not None:
                expected=int(headers['CMR-Hits'])
            batch=payload['feed']['entry']; entries.extend(batch)
            if len(batch)<200:break
            page+=1
        assert len({e['id'] for e in entries})==len(entries), 'Duplicate catalogue IDs: review pagination.'
        if expected is not None:
            assert len(entries)==expected, 'Catalogue changed or paging was incomplete. Requery this month.'
        all_entries[(month,product)] = entries
        summary.append({'month':month,'role':role,'product':product,
                        'catalogue_candidates':len(entries),'pixel_coverage_verified':False})
        print(month,product,len(entries),'catalogue candidates')
summary_df=pd.DataFrame(summary)
summary_df.to_csv(RUN/'inventory/pilot_inventory.csv',index=False)
display(summary_df.pivot(index=['month','role'],columns='product',values='catalogue_candidates'))
print('Saved inventory to',RUN/'inventory')


2001-01 MOD14A1 20 catalogue candidates
2001-01 MYD14A1 0 catalogue candidates
2001-01 MOD14 76 catalogue candidates
2001-01 MYD14 0 catalogue candidates
2001-01 MOD03 76 catalogue candidates
2001-01 MYD03 0 catalogue candidates
2002-01 MOD14A1 20 catalogue candidates
2002-01 MYD14A1 0 catalogue candidates
2002-01 MOD14 75 catalogue candidates
2002-01 MYD14 0 catalogue candidates
2002-01 MOD03 75 catalogue candidates
2002-01 MYD03 0 catalogue candidates
2022-12 MOD14A1 20 catalogue candidates
2022-12 MYD14A1 20 catalogue candidates
2022-12 MOD14 79 catalogue candidates
2022-12 MYD14 79 catalogue candidates
2022-12 MOD03 80 catalogue candidates
2022-12 MYD03 79 catalogue candidates
2023-12 MOD14A1 20 catalogue candidates
2023-12 MYD14A1 20 catalogue candidates
2023-12 MOD14 77 catalogue candidates
2023-12 MYD14 78 catalogue candidates
2023-12 MOD03 77 catalogue candidates
2023-12 MYD03 78 catalogue candidates


,product,MOD03,MOD14,MOD14A1,MYD03,MYD14,MYD14A1
month,role,,,,,,
2001-01,disputed,76,76,20,0,0,0
2002-01,control,75,75,20,0,0,0
2022-12,control,80,79,20,79,79,20
2023-12,disputed,77,77,20,78,78,20


Saved inventory to /content/drive/MyDrive/emberline_clean_rebuild/runs/20261114T163518Z_5b1b1c54/inventory


## 8. Stage 1 checkpoint — stop here and inspect the inventory

Share the printed `RUN_ID` and the final inventory table. Do not share authentication output or tokens. The next section will download a small verified batch, inspect actual HDF dataset metadata and build daily observation accounting before deriving fire totals.

The pixel-processing stage must handle a known issue from this pilot: December and January eight-day packages can contain the same January days. Compare fire-mask and QA bytes before deduplicating identical daily layers; conflicting layers require investigation. Never sum both copies.

A second known issue: a December 21, 2022 Terra swath is listed in LAADS but absent from CMR's fire-product query. Reconcile missing catalogue matches against the direct archive before declaring an unrecoverable outage.

## Remaining rebuild stages

1. Verify downloads, hashes, byte sizes, dataset dimensions, dates and source versions; preserve immutable raw files and a resumable acquisition manifest.
2. Read fire, nonfire, cloud, unknown, missing-input and QA states. Reconcile swaths against geolocation and outage/recovery records. Measure regional and local observation support, including quiet areas.
3. Rebuild native detections and sensor-specific H3 cell-days with explicit sensor and confidence provenance. Validate study boundaries and administrative support. Keep missing values distinct from observed zeros.
4. Evaluate S-NPP/NOAA-20 replacement on independently matched common observation support, including jointly observed no-detection cases. Test deduplication and replacement uncertainty before filling anything.
5. Validate Terra → combined MODIS → VIIRS transfer and district/custom-area allocation. Retain unavailable outputs when support or uncertainty is not defensible. Fit any learned category rules and model selection without outer-test leakage.
6. Freeze repaired inputs. Rerun nested regional errors, empirical interval coverage, district allocation and paired training-window sensitivity. Test annual totals and rankings separately. A marginal confidence-slice interval cannot certify a summed-filter interval.
7. Export a versioned staging release in the existing frontend contract, run actual-data checks, then replace browser datasets only after review. Earlier scores remain tied to their earlier dataset.

## Existing frontend contract to preserve

- `manifest.json`, per-region `monthly.json`, `report.json`, `sub-<id>.json`, boundary and subdivision GeoJSON, `members.json`, `hexagons.json`, `map-<year>.json`, and existing comparison assets.
- Monthly required fields: `month` (`YYYY-MM`), `cat`, `conf`, `modis_raw`, `viirs_snpp_raw`, `harmonized`, `lo`, `hi`, `source`, `period_status`, `comparison_eligible`, `low_data`.
- Exactly 12 unique month/category/confidence rows per month. JSON uses `null` for unavailable values; no NaN/Infinity or silent partial sums.
- Preserve administrative IDs, H3 resolution and the established index mappings. Any revised spatial support must be checked against current area selections.
- Keep provenance and richer quality metadata in the canonical data and additional export fields. A replacement must not be labelled as an observed S-NPP measurement. If unchanged UI text cannot truthfully represent a method, hold that method from release until the conflict is resolved.

This notebook is a Stage 1 starter, not a completed or scientifically validated rebuild pipeline.
